# Part 4: Build the Best Classifier

In this notebook, you will build the best classifier you can for either the **binary** or **multiclass** task from the previous notebooks.

- **Binary task**: Predict whether a bill was assigned to the "Housing and Economic Development" committee (`data/y.json`)
- **Multiclass task**: Predict which committee a bill was assigned to (`data/y_multi.json`)

You may use any scikit-learn estimator, pipeline, or preprocessing technique. You are not required to implement anything from scratch.

**Grading**: Your code must run end-to-end without errors, and your written responses must be substantive and demonstrate your reasoning. Raw performance numbers are not graded — we care about your process and justification.

**It may be tempting to spend a lot of time on this question trying to eke out the best possible performance. Don't do that! Commit to a model and tune the hyperparameters, then focus on writing your analysis.**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore') 

# 1. Load Data (Binary Task: Housing vs Not Housing)
df_X = pd.read_json('data/X.json')
y = pd.read_json('data/y.json')['committee_bool'].values
X_text = np.stack(df_X['text_embedding'].values)

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=6140, stratify=y
)

# 3. Pipeline Design: StandardScaler + Logistic Regression with Balanced Class Weights
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=6140))
])

# 4. Hyperparameter Tuning using GridSearchCV
param_grid = {
    'lr__C': [0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l2']
}

# Optimizing for 'recall' since missing a Housing bill is worse than a false positive in our scenario
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid.best_params_}")

# 5. Evaluation
y_pred = grid.predict(X_test)
print("\nClassification Report on Test Set:")
print(classification_report(y_test, y_pred, zero_division=0))


Best Hyperparameters: {'lr__C': 0.01, 'lr__penalty': 'l2'}

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97       241
           1       0.56      0.95      0.70        20

    accuracy                           0.94       261
   macro avg       0.78      0.94      0.83       261
weighted avg       0.96      0.94      0.95       261



## Written Responses

Answer each question below. Aim for 2-4 sentences per question — be specific and reference your results where relevant.

### Task & Setup

**Q1. Which task did you pick (binary or multiclass), and why?**

I decided to take the binary classification (predicting whether a bill belongs to the 'Housing and Economic Development' committee). I chose this because optimizing a model for a specific, high-priority real-world use case, like building an automated flagging system for a specialized review team, felt much more practical than trying to build a perfect "sorting hat" for all 17 highly overlapping committees. 

**Q2. What preprocessing steps did you apply to the features, if any? Why did you make those choices?**

I chose to only use the text_embedding features rather than the titles. Based on my results in Notebook 2, bill titles are often too vague, whereas the full text embeddings capture the rich semantic vocabulary that we actually need. For preprocessing, I used a StandardScaler inside my Pipeline. This ensures all 384 dimensions have a mean of 0 and a variance of 1, which is crucial so that dimensions with naturally larger numeric ranges don't unfairly dominate the gradient calculations in Logistic Regression.


### Model Selection

**Q3.** Which model(s) did you try? Which did you select as your final model, and why did it outperform the others?

I actually tried a `RandomForestClassifier` first, assuming a complex ensemble tree would perform best. However, it severely underperformed on the minority class (only hitting a 0.10 Recall for Housing bills), likely because the dataset is so imbalanced. I switched back to a tuned `LogisticRegression` model but initialized it with `class_weight='balanced'`. This drastically outperformed the Random Forest by directly penalizing the model much heavier for missing the rare Housing bills. It reached an 0.95 Recall on the test set.


### Hyperparameter Tuning
**Q4. How did you tune your model's hyperparameters? What values or settings worked best?**

I used `GridSearchCV` with 5-fold cross-validation to search over the C parameter (inverse of regularization strength) for Logistic Regression, testing `[0.01, 0.1, 1, 10, 100]`. The most important thing I did here was setting the grid's scoring parameter to 'recall' instead of the default accuracy, forcing the grid to find the model that catches the most housing bills. The optimal hyperparameters were `C = 0.01` and `penalty = 'l2'`. This means a relatively strong amount of L2 regularization helped the model generalize better to the test set and maximize retrieval.


### Evaluation
**Q5. Which metric(s) did you use to evaluate your model? Why are they appropriate for your chosen task and dataset?**

As I mentioned in Notebook 2, my primary evaluation metric for this binary task is Recall. Since we are essentially building an automated system to flag Housing bills for human to review, the worst-case scenario is a False Negative, a relevant bill slipping by unnoticed. My tuned model achieved a 95% recall for the positive class, meaning it successfully caught almost every single housing bill. Precision did drop to 56%, meaning human reviewers will have to toss out a few false alarms, but that is a perfectly acceptable trade-off for this scenario.


### Reflection
**Q6. What would you try next if you had more time? Are there modeling choices, features, or techniques you were curious about but didn't get to?**

If I had more time, I would love to try combining the title_embedding and text_embedding into a single fused feature vector to see if the title provides any unique supplementary signal that the body text misses. I also really wanted to experiment with oversampling techniques like SMOTE (Synthetic Minority Over-sampling Technique). If I could algorithmically generate synthetic "Housing" embeddings during training, I might be able to boost the Precision score back up without sacrificing the Recall value I got here.
